## Prep

### Imports, shared definitions, datasets

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily
import pointpats
import numpy as np

In [ ]:
listings_url = "https://data.insideairbnb.com/spain/catalonia/barcelona/2025-09-14/data/listings.csv.gz"
listings_df = pd.read_csv(listings_url, compression='gzip')
listings_df

## Part 2

In [ ]:
listings_geometry = gpd.points_from_xy(listings_df['longitude'], listings_df['latitude'], crs="EPSG:4326")
listings_geometry

In [ ]:
listings_gdf = gpd.GeoDataFrame(listings_df, geometry=listings_geometry)
listings_gdf.head()

In [ ]:
listings_gdf.explore(tiles="CartoDB Positron")

In [ ]:
# density

In [ ]:
f, ax = plt.subplots()
listings_gdf.plot(ax=ax, markersize=0.05)

In [ ]:
# Q: Are the Airbnb listings distributed equally across the city?
# No

f, ax = plt.subplots()
# listings_gdf.plot(ax=ax, markersize=0.05)
pointpats.plot_density(
    listings_gdf,
    bandwidth=500,
    levels=25,
    # alpha=0.55,
    # cmap="magma_r",
    # linewidths=1,
    ax=ax,
)
contextily.add_basemap(
    ax=ax,
    crs=listings_gdf.crs,
    source="CartoDB Positron No Labels",
)

In [ ]:
# Does it depend on the type of listing or its price?

In [ ]:
listings_gdf.head()

In [ ]:
listings_gdf.columns


In [ ]:
listings_gdf["property_type"].head(20)

In [ ]:
listings_gdf.explore("property_type")

In [ ]:
listings_gdf["price"].head(10)

In [ ]:
listings_gdf["price_dollars"] = (
    listings_gdf["price"]
      .str.replace(r"[\$,]", "", regex=True)
      .astype(float)
)

In [ ]:
listings_gdf["price_dollars"].head(10)

In [ ]:
listings_gdf.explore("price_dollars")

In [ ]:
listings_gdf["price_dollars_log"] = np.log(listings_gdf["price_dollars"])

In [ ]:
listings_gdf.explore("price_dollars_log")

# Thoughts / Plan of attack

Ok, answering "Does it depend on the type of listing or its price?" is not straightforward (at least for me) because "type of listing" is categorical whilst price is numerical.

Based on reading through bits of course again, summary of planned approach:

## price

1. Find all points which have a price defined (ignore NaN)
2. Get the convex cull of the points
3. Apply a hexbin over this area, at some approx radius
4. Map each price to its hexbin
5. Compute median price of each hexbin (using median here as there are some very large prices)
6. For hexbin, apply spatial autocorrelation, Moran's I?

## type of listing

1. Find all points which have a listing type
2. For each point, find it's nearest neighbour (using KNN); perhaps choose radius as same as approx radius of hexbin for price?
3. Compute a binary outcome by checking if each point has the same category as its neighbhour
4. Using Join Counts statistic from here